# CDM catalog scan

Lists every table and view in `clhhd01p_clh` with its columns (metadata only, no data scanned), tags columns by the themes our open problems need, and shortlists candidates to look at closely.

Themes sought: product / account holdings (F5), TM services, balances, credit facilities, credit risk, cash (F4), geography (F2), account events (open / close / status, for attrition), revenue, coverage team. Link keys: power id, cust_dim_key, party_dim_key, relationship, account keys. Dimension tables are listed separately: we need them to decode keys such as pd_key.

Outputs: one HTML per cell; full column list and table summary saved as CSV.

In [ ]:
# ============================================================
# 1. Setup and config
# ============================================================
import os, re, html, warnings, datetime as _dt
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore")

# --- NumPy alias shim (old pyspark code paths use removed aliases) ---
import numpy as np
for _a, _t in (("object0", object), ("bool8", np.bool_), ("int0", np.intp), ("uint0", np.uintp),
               ("bool", bool), ("object", object)):
    if not hasattr(np, _a):
        setattr(np, _a, _t)

# --- pandas shim: PySpark 3.3 createDataFrame calls iteritems (removed in pandas 2.0) ---
import pandas as pd
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

from IPython.display import display, HTML
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark import StorageLevel

spark = (SparkSession.builder
         .appName("cdm_customer_fact_explore")
         .config("spark.ui.showConsoleProgress", "false")   # no stage progress bars in cell outputs
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:  # silence Atlas lineage hook + harmless executor-release noise (log4j2 on Spark 3.3)
    _L = spark._jvm.org.apache.logging.log4j
    for n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
              "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
              "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(n, _L.Level.OFF)
except Exception:
    pass
# Unparseable date strings -> NULL instead of SparkUpgradeException (we probe many formats)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

# ---------------- config ----------------
CDM_DB          = "clhhd01p_clh"
SIBLING_DB_LIKE = "*clh*"     # other databases whose name matches, listed by name only
SHORTLIST_N     = 30
OUT_DIR         = "../explore/cdm_catalog"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- helpers ----------------
CSS = """<style>
.cx {font-family: Arial, sans-serif; font-size: 12px;}
.cx h3 {margin: 14px 0 4px 0; font-size: 14px;}
.cx h4 {margin: 10px 0 3px 0; font-size: 13px; color: #1f3b63;}
.cx table {border-collapse: collapse; margin: 2px 0 8px 0;}
.cx th {background: #e8eef6; border: 1px solid #c8d2e0; padding: 2px 6px; text-align: left; white-space: nowrap;}
.cx td {border: 1px solid #dde3ec; padding: 2px 6px; vertical-align: top; max-width: 420px; overflow-wrap: anywhere;}
.cx .note {color: #555; margin: 2px 0;}
.cx .warn {color: #a33; font-weight: bold;}
.cx pre {background: #f6f8fa; border: 1px solid #dde3ec; padding: 6px; white-space: pre-wrap; font-size: 11px; max-width: 1400px;}
</style>"""

def esc(x):
    return html.escape("" if x is None else str(x))

def tbl(pdf, max_rows=None):
    """pandas DataFrame -> compact HTML table (values escaped)."""
    if pdf is None or len(pdf) == 0:
        return "<p class='note'>(no rows)</p>"
    if max_rows is not None:
        pdf = pdf.head(max_rows)
    head = "".join(f"<th>{esc(c)}</th>" for c in pdf.columns)
    body = "".join("<tr>" + "".join(f"<td>{esc(v)}</td>" for v in row) + "</tr>"
                   for row in pdf.astype(object).where(pdf.notna(), "").itertuples(index=False))
    return f"<table><tr>{head}</tr>{body}</table>"

def show(parts):
    """One consolidated HTML output per cell; tex2jax_ignore keeps any dollar sign from turning into math."""
    display(HTML(CSS + "<div class='cx tex2jax_ignore mathjax_ignore'>" + "".join(parts) + "</div>"))

def fnum(x):
    if x is None:
        return ""
    try:
        x = float(x)
    except Exception:
        return str(x)
    if np.isnan(x):
        return ""
    a = abs(x)
    for d, s in ((1e12, "T"), (1e9, "B"), (1e6, "M"), (1e3, "K")):
        if a >= d:
            return f"{x / d:.2f}{s}"
    if x == int(x):
        return str(int(x))
    return f"{x:.4g}"

def fpct(x, digits=1):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return f"{100 * float(x):.{digits}f}%"

def qcol(c):
    return F.col(f"`{c}`")

NUMERIC_T = (T.ByteType, T.ShortType, T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.DecimalType)
COMPLEX_T = (T.ArrayType, T.MapType, T.StructType, T.BinaryType)
DATE_NAME = re.compile(r"(_dt$|_dte$|_date$|^date_|_date_|_ts$|_tmstmp|timestamp|^dt_|_dt_|asof|as_of|snap|_month$|_mo$|_yrmo|yyyymm|period)")

def kind_of(dtype):
    if isinstance(dtype, (T.DateType, T.TimestampType)): return "date"
    if isinstance(dtype, T.BooleanType): return "bool"
    if isinstance(dtype, NUMERIC_T): return "num"
    if isinstance(dtype, COMPLEX_T): return "complex"
    return "str"

def is_date_like(name, dtype):
    k = kind_of(dtype)
    if k == "date":
        return True
    return k in ("str", "num") and bool(DATE_NAME.search(name.lower())) \
        and not isinstance(dtype, (T.FloatType, T.DoubleType))

def date_expr(c, dtype):
    """Parse a date-like column of any common storage format to DATE (NULL if unparseable)."""
    col = qcol(c)
    if isinstance(dtype, T.DateType):
        return col
    if isinstance(dtype, T.TimestampType):
        return F.to_date(col)
    s = F.trim(col.cast("string"))
    return (F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}"), F.to_date(F.substring(s, 1, 10), "yyyy-MM-dd"))
             .when(s.rlike(r"^\d{8}$"), F.to_date(s, "yyyyMMdd"))
             .when(s.rlike(r"^\d{1,2}/\d{1,2}/\d{4}"),
                   F.to_date(F.regexp_extract(s, r"^(\d{1,2}/\d{1,2}/\d{4})", 1), "M/d/yyyy"))
             .when(s.rlike(r"^\d{6}$"), F.to_date(F.concat(s, F.lit("01")), "yyyyMMdd"))
             .when(s.rlike(r"^\d{4}-\d{2}$"), F.to_date(F.concat(s, F.lit("-01")), "yyyy-MM-dd")))

def real_date(p):
    """Drop placeholder dates (year <= 1900 or >= 2100, e.g. 0001-01-01 / 9999-12-31) for min/max."""
    return F.when(F.year(p).between(1901, 2099), p)

show([f"<h3>Setup</h3><p>Spark {esc(spark.version)} · pandas {esc(pd.__version__)} · numpy {esc(np.__version__)}</p>",
      tbl(pd.DataFrame([("CDM_DB", CDM_DB), ("SIBLING_DB_LIKE", SIBLING_DB_LIKE), ("OUT_DIR", os.path.abspath(OUT_DIR))],
                       columns=["setting", "value"]))])


In [ ]:
# ============================================================
# 2. Scan the catalog (metadata only)
# ============================================================
THEMES_SCAN = {
    "products / accounts": r"(prod|product|(^|_)acct|account|arrngmt|arrangement|holding|(^|_)svc|service|portfolio)",
    "TM services":         r"((^|_)tm(_|$)|treasury|lockbox|(^|_)ach|wire|(^|_)rtp|pos_?pay|positive|(^|_)rdc|remote_dep|sweep|(^|_)zba|(^|_)ccr|card|merchant|payabl|receiv|pinacle|pinnacle)",
    "balances":            r"((^|_)bal(_|$)|balance|(^|_)adb|avg_bal|ledger|collected|deposit|(^|_)dda|(^|_)mmda|saving)",
    "credit facilities":   r"(loan|facilit|commit|outst|(^|_)os_|matur|(^|_)line|undrawn|unfund|oblig|(^|_)lc(_|$)|lease|collat|draw|(^|_)dhe|(^|_)ead)",
    "credit risk":         r"((^|_)pd(_|$)|rating|(^|_)rtg|grade|default|watch|critic|classif|delinq|past_?due|non_?accr|chg_?off|charge_?off|(^|_)lgd|(^|_)el_|impair|bankrupt|covenant)",
    "cash":                r"(cash|currency|(^|_)atm|teller|vault|(^|_)ctr(_|$)|coin|branch_dep|night_dep)",
    "geography":           r"(addr|(^|_)zip|postal|city|(^|_)st_cd|state_cd|(^|_)state|county|cnty|(^|_)msa|(^|_)cbsa|latit|longit|(^|_)lat(_|$)|(^|_)lon(_|$)|geo|census|cntry|country)",
    "account events":      r"(open_dt|opn_dt|close|clos_|closed|attrit|churn|status|stat_cd|(^|_)exit|termin|inactiv|dormant|reopen)",
    "revenue / profit":    r"((^|_)rev(_|$)|revenue|(^|_)fee|income|profit|(^|_)ep_|(^|_)nii|spread|margin)",
    "coverage team":       r"(empl|officer|(^|_)rm_|(^|_)rm$|(^|_)tmo|banker|coverage)",
}
SOUGHT = list(THEMES_SCAN)
_TRX = {k: re.compile(v) for k, v in THEMES_SCAN.items()}
LINK = {"power id": re.compile(r"pwr_id|power_id"), "cust_dim_key": re.compile(r"^cust_dim_key$"),
        "party_dim_key": re.compile(r"^party_dim_key$"), "relationship": re.compile(r"^rltn_"),
        "account key": re.compile(r"(^|_)acct(_\w*)?(_key|_id|_nbr|_no)$|account_(key|id|nbr)")}
GRAIN = re.compile(r"(^as_of_dt$|^as_of_date$|^time_key$|^month$|^mnth$|^mo_|^snap|^rpt_dt|^busn_dt|^bus_dt|^eff_dt|_yrmo$|^yrmo|^period|^txn_dt$|^post_dt$|^tran_dt$)")

def parse_describe(d):
    cols, part, info, sec = [], [], {}, "cols"
    for name, dtype, cmt in d[["col_name", "data_type", "comment"]].itertuples(index=False):
        name = (name or "").strip()
        if name.startswith("#"):
            sec = "part" if "Partition" in name else ("info" if "Detailed" in name else sec)
            continue
        if name == "":
            if sec == "cols": sec = "gap"
            continue
        if sec == "cols": cols.append((name, (dtype or "").strip(), (cmt or "").strip()))
        elif sec == "part": part.append(name)
        elif sec == "info": info[name] = (dtype or "").strip()
    return cols, part, info

tabs = spark.sql(f"SHOW TABLES IN {CDM_DB}").toPandas()
tabs = tabs[~tabs["isTemporary"].astype(bool)]
try:
    sib = spark.sql(f"SHOW DATABASES LIKE '{SIBLING_DB_LIKE}'").toPandas().iloc[:, 0].tolist()
except Exception:
    sib = []

tsum, call, errs = [], [], []
for t in sorted(tabs["tableName"]):
    full = f"{CDM_DB}.{t}"
    try:
        cols, part, info = parse_describe(spark.sql(f"DESCRIBE TABLE EXTENDED {full}").toPandas())
    except Exception as e:
        errs.append({"table": t, "error": str(e).split("\n")[0][:200]})
        continue
    names = [c for c, _, _ in cols]
    hits = {k: [c for c in names if rx.search(c.lower())] for k, rx in _TRX.items()}
    links = [k for k, rx in LINK.items() if any(rx.search(c.lower()) for c in names)]
    grain = [c for c in names if GRAIN.search(c.lower())]
    vt = info.get("View Text") or info.get("View Original Text") or ""
    srcs = sorted(set(m.lower() for m in re.findall(r"\b(?:from|join)\s+`?([A-Za-z_]\w*\.[A-Za-z_]\w*)`?", vt, flags=re.I)))
    n_sought = sum(1 for k in SOUGHT if hits[k])
    tsum.append({"table": t, "type": info.get("Type", ""), "cols": len(cols),
                 "is_dim": bool(re.search(r"(_dim(_|$)|^dim_|_lkp|lookup|_ref(_|$)|_xref|_cd_)", t.lower())),
                 "links": ", ".join(links), "grain": ", ".join(grain[:3]), "partitions": ", ".join(part),
                 "stats": info.get("Statistics", ""), "sources": ", ".join(srcs),
                 "themes": " · ".join(f"{k} {len(v)}" for k, v in hits.items() if v),
                 "examples": ", ".join(dict.fromkeys(c for k in SOUGHT for c in hits[k][:3]))[:300],
                 "n_themes": n_sought, "n_hits": sum(len(v) for v in hits.values()),
                 "score": n_sought + (2 if links else 0)})
    call += [{"table": t, "column": c, "type": ty, "comment": cm,
              "themes": ", ".join(k for k, rx in _TRX.items() if rx.search(c.lower()))} for c, ty, cm in cols]

TS = pd.DataFrame(tsum)
CA = pd.DataFrame(call)
TS.to_csv(os.path.join(OUT_DIR, "cdm_tables.csv"), index=False)
CA.to_csv(os.path.join(OUT_DIR, "cdm_columns.csv"), index=False)
if len(TS) == 0:
    raise RuntimeError(f"No readable tables in {CDM_DB} ({len(errs)} errors)")

n_views = int((TS["type"].str.upper() == "VIEW").sum())
facts = pd.DataFrame([("Database", CDM_DB), ("Tables and views", f"{len(TS):,} ({n_views:,} views)"),
                      ("Columns", f"{len(CA):,}"), ("Dimension / lookup tables", int(TS["is_dim"].sum())),
                      ("With a customer or account link key", int((TS["links"] != "").sum())),
                      ("Could not describe", len(errs)),
                      ("Other databases like " + SIBLING_DB_LIKE, ", ".join(x for x in sib if x != CDM_DB) or "(none)")],
                     columns=["item", "value"])
short = (TS[~TS["is_dim"] & (TS["n_themes"] > 0)]
         .sort_values(["score", "n_hits"], ascending=False).head(SHORTLIST_N))
show_cols = ["table", "type", "cols", "links", "grain", "themes", "examples"]
dims = TS[TS["is_dim"]].sort_values("table")
dims_v = dims[["table", "cols"]].assign(columns=[", ".join(CA.loc[CA["table"] == t, "column"].head(10)) for t in dims["table"]])
rest = TS.sort_values("table")[["table", "type", "cols", "links", "themes"]]

show(["<h3>CDM catalog scan</h3>", tbl(facts),
      f"<h4>Shortlist — top {len(short)} non-dimension tables by themes sought (+2 if a customer/account key)</h4>",
      tbl(short[show_cols]),
      f"<h4>Dimension / lookup tables — {len(dims)}</h4>", tbl(dims_v),
      f"<h4>All tables — {len(rest)}</h4>", tbl(rest)]
     + ([f"<h4>Could not describe — {len(errs)}</h4>", tbl(pd.DataFrame(errs))] if errs else [])
     + [f"<p class='note'>Saved: {esc(os.path.abspath(os.path.join(OUT_DIR, 'cdm_tables.csv')))} and cdm_columns.csv</p>"])
